# Fine-tuning NOTAM (QLoRA Qwen3)
Jalankan sel dari atas ke bawah. Menu **Runtime > Change runtime type > T4 GPU** dulu.

In [ ]:
!nvidia-smi | head -12

## 1. Unggah `finetune-colab.zip`
Berisi `train.jsonl`, `val.jsonl`, `train_lora.py`, `merge_export.py`. Jalankan sel, lalu pilih file zip.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()
for n in up: zipfile.ZipFile(n).extractall('.')
os.makedirs('out', exist_ok=True)
!mv -f train.jsonl val.jsonl out/ 2>/dev/null
!ls -la . out

## 2. Pasang paket

In [ ]:
!pip -q install -U "transformers>=4.51" peft trl datasets bitsandbytes accelerate

## 3. Latih
T4 (16 GB) cukup untuk Qwen3-8B. Ubah `MODEL` ke `Qwen/Qwen3-4B` jika kehabisan memori. Perkiraan 1–2 jam.

In [ ]:
MODEL = 'Qwen/Qwen3-8B'
!python train_lora.py --model $MODEL --max-len 2048 --out out/lora

## 4. Simpan adaptor ke Google Drive (cadangan)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/notam-ft && cp -r out/lora /content/drive/MyDrive/notam-ft/

## 5. Konversi adaptor ke GGUF
Tanpa menggabungkan model (RAM Colab gratis tidak cukup untuk 8B). Ollama menempelkan adaptor ke `qwen3:8b`.

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip -q install -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python llama.cpp/convert_lora_to_gguf.py out/lora --base-model-id Qwen/Qwen3-8B --outfile notam-lora.gguf --outtype f16
!ls -la notam-lora.gguf && cp notam-lora.gguf /content/drive/MyDrive/notam-ft/

## Di PC
Unduh `notam-lora.gguf` dari Drive. Buat `Modelfile`:
```
FROM qwen3:8b
ADAPTER ./notam-lora.gguf
PARAMETER temperature 0
```
Lalu `ollama create notam-ft -f Modelfile`.